# 🤖 Machine Learning Made Simple – Part 2
### Chen Hajaj | Missing Values, Encoding & Feature Scaling

--

## ⚙️ Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split

# Style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlepad'] = 10
print('✅ All set!')


---
# 🕵️ Part 1: Missing Data – A Crime Investigation

### What is Dirty Data?

| Problem Type | Description | Example |
|---|---|---|
| **Missing Values** 🟠 | Values that are completely absent | NaN, None |
| **Outliers** 🔴 | Extreme values far from the rest | age = 999 |
| **Duplicates** 🟣 | Repeated rows in the dataset | Same user appears twice |
| **Bad Values** 🩵 | Corrupt or inconsistent values | N/A, NULL, -, wrong character |

In [ ]:
# 🎯 Activity: Find the problems!
# Look at this table for 10 seconds — how many issues can you spot?

dirty_data = pd.DataFrame({
    'name':    ['Alice', 'Bob',  'Carol', 'Bob',  'Dana',  'Evan'],
    'age':     [25,      None,   30,      26,     999,     22   ],
    'salary':  [8000,    9000,   None,    9000,   7500,    'N/A'],
    'country': ['US',    'US',   'UK',    'US',   'ger',   'US' ]
})

print('📋 Our dataset:')
display(dirty_data)
print(f'\n🔍 How many missing values? {dirty_data.isnull().sum().sum()}')
print(f'🔍 How many duplicate rows? {dirty_data.duplicated().sum()}')

---
## 🔎 The Three Types of Missingness

| Type | Full Name | When does it happen? | Real-world example |
|---|---|---|---|
| **MCAR** | Missing Completely at Random | Missing value is unrelated to anything | Each participant answered a different subset of questions |
| **MAR** | Missing at Random | Missingness depends on *another* variable | Men less likely to report income |
| **MNAR** | Missing Not at Random | Missingness depends on the *missing value itself* | High earners less likely to report their income |

In [ ]:
# Visualization: MNAR – The Paradox!
np.random.seed(42)
n = 200
salary = np.random.normal(15000, 5000, n)

# MNAR: the higher the salary, the less likely it is to be reported
prob_missing = 1 / (1 + np.exp(-(salary - 18000) / 2000))
is_missing = np.random.random(n) < prob_missing

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(range(n), salary, c=['red' if m else 'steelblue' for m in is_missing],
                alpha=0.7, s=30)
axes[0].set_title('Missing values by salary level (MNAR)', fontweight='bold')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Salary ($)')
axes[0].legend(handles=[
    mpatches.Patch(color='red', label='Missing'),
    mpatches.Patch(color='steelblue', label='Present')
])

axes[1].hist(salary[~is_missing], bins=30, color='steelblue', alpha=0.7, label='Reported salary')
axes[1].hist(salary[is_missing], bins=30, color='red', alpha=0.5, label='Unreported salary')
axes[1].set_title('High salary = less likely to be reported!', fontweight='bold')
axes[1].set_xlabel('Salary ($)')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Mean reported salary:  ${salary[~is_missing].mean():,.0f}')
print(f'True mean salary:      ${salary.mean():,.0f}')
print(f'Bias: ${salary[~is_missing].mean() - salary.mean():,.0f} (the reported data is skewed!)')

---
# 💉 Part 2: Missing Value Imputation
### 6 Methods – Which is Best?

In [ ]:
# Dataset with missing values
np.random.seed(0)
original = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100], dtype=float)
data_with_missing = original.copy()
data_with_missing[[2, 5, 8]] = np.nan  # missing at indices 2, 5, 8

df = pd.DataFrame({'score': data_with_missing})

print('📋 Data with missing values:')
print(df.T.to_string())

# 6 methods
methods = {}

# 1. Listwise Deletion
methods['Listwise Deletion'] = df.dropna()['score'].values

# 2. Mean
imp_mean = SimpleImputer(strategy='mean')
methods['Mean'] = imp_mean.fit_transform(df)[:, 0]

# 3. Median
imp_med = SimpleImputer(strategy='median')
methods['Median'] = imp_med.fit_transform(df)[:, 0]

# 4. Constant
imp_const = SimpleImputer(strategy='constant', fill_value=0)
methods['Constant (0)'] = imp_const.fit_transform(df)[:, 0]

# 5. Forward Fill
methods['Forward Fill'] = df.ffill()['score'].values

# 6. KNN — include row index as a feature so nearest neighbors are found by position
df_knn = pd.DataFrame({'score': data_with_missing, 'idx': np.arange(len(df), dtype=float)})
imp_knn = KNNImputer(n_neighbors=2)
methods['KNN (k=2)'] = imp_knn.fit_transform(df_knn)[:, 0]

# Visual comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
missing_idx = [2, 5, 8]
present_idx = [i for i in range(10) if i not in missing_idx]

for i, (name, values) in enumerate(methods.items()):
    ax = axes[i]
    if name == 'Listwise Deletion':
        ax.plot(range(len(values)), values, 'o-', color='gray')
        ax.set_title(name, fontweight='bold')
        ax.set_xlabel('After deletion')
    else:
        ax.plot(range(10), original, 'o--', color='lightgray', label='True value', alpha=0.5)
        ax.plot(present_idx, values[present_idx], 'o', color='steelblue', label='Present')
        ax.plot(missing_idx, values[missing_idx], 's', color='red', markersize=10, label='Imputed')
        ax.set_title(name, fontweight='bold')
        ax.legend(fontsize=9)
    ax.set_ylabel('Score')
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparing Imputation Methods', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


---
# 🏷️ Part 3: Encoding – Numbers from Categories

### Why do we need Encoding?
> ML algorithms work with **numbers**. `France`, `Germany`, `Spain` are not numbers!

In [ ]:
# 🎯 Activity: Label Encoding – The Hidden Problem
countries = pd.DataFrame({'country': ['France', 'Spain', 'Germany', 'Spain', 'France', 'Germany']})

le = LabelEncoder()
countries['Label Encoded'] = le.fit_transform(countries['country'])

print('🏷️ Label Encoding:')
display(countries)
print()
print('⚠️  The problem: according to the encoding, Germany(1) > France(0)')
print('   The algorithm thinks Germany is "greater" than France!')
print('   This is only OK if there is a natural order (Ordinal): small < medium < large')

In [ ]:
# One-Hot Encoding – The Solution
one_hot = pd.get_dummies(countries['country'], prefix='country')
result = pd.concat([countries['country'], one_hot], axis=1)

print('One-Hot Encoding:')
display(result)

# Visualization of the Living Table
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
tbl = ax.table(
    cellText=result.values,
    colLabels=result.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1.5, 2)

# Highlight 1s in green
for (row, col), cell in tbl.get_celld().items():
    if row > 0 and col > 0:
        val = result.iloc[row-1, col]
        cell.set_facecolor('#c8e6c9' if val == 1 else '#ffffff')

plt.title('One-Hot Encoding – Living Table', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# ⚠️ Dummy Trap!
print('With all columns (Dummy Trap):')
print('   France + Germany + Spain = always 1')
print('   One column is always derivable from the others → Multicollinearity!')
print()

one_hot_drop = pd.get_dummies(countries['country'], prefix='country', drop_first=True)
result_drop = pd.concat([countries['country'], one_hot_drop], axis=1)

print('With drop_first=True (the fix):')
display(result_drop)
print('   If country_Germany=0 and country_Spain=0 → it must be France! No extra column needed.')

---
# ⚖️ Part 4: Feature Scaling

### The Problem: Features on very different scales!

| Feature | Range |
|---|---|
| Age | 0–100 |
| Salary | 0–100,000 |
| Score | 0–100 |

Algorithms like KNN and SVM will treat salary as **far more important** just because of its scale!

In [ ]:
# Comparing three scaling methods
np.random.seed(42)
data = pd.DataFrame({
    'age':    np.random.randint(20, 60, 100),
    'salary': np.random.randint(5000, 30000, 100)
})

scalers = {
    'Original (No Scaling)':   data,
    'StandardScaler (Z-score)': pd.DataFrame(StandardScaler().fit_transform(data), columns=data.columns),
    'MinMaxScaler (0-1)':       pd.DataFrame(MinMaxScaler().fit_transform(data), columns=data.columns),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, scaled) in zip(axes, scalers.items()):
    ax.scatter(scaled['age'], scaled['salary'], alpha=0.5, color='steelblue')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Age')
    ax.set_ylabel('Salary')
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparing Scaling Methods', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('StandardScaler: mean=0, std=1 for every feature')
print('MinMaxScaler: all values between 0 and 1')


---
# 😱 One Outlier Breaks Everything!

### Imagine: one employee earning $1,000,000 joins a company of 50 people...

In [ ]:
np.random.seed(42)
normal_salaries = np.random.normal(12000, 2000, 50)

# Add one outlier
with_outlier = np.append(normal_salaries, 1_000_000)

print('🏢 Company with 50 employees:')
print(f'   Mean salary:   ${normal_salaries.mean():,.0f}')
print(f'   Median salary: ${np.median(normal_salaries):,.0f}')
print()
print('😱 One employee earning $1,000,000 joins:')
print(f'   Mean salary:   ${with_outlier.mean():,.0f}  ← jumped by {(with_outlier.mean()-normal_salaries.mean())/normal_salaries.mean()*100:.0f}%!')
print(f'   Median salary: ${np.median(with_outlier):,.0f}   ← slightly changed')
print()

# Scaling arrays
data_normal  = normal_salaries.reshape(-1, 1)
data_outlier = with_outlier.reshape(-1, 1)

std_normal     = StandardScaler().fit_transform(data_normal)
std_outlier    = StandardScaler().fit_transform(data_outlier)
minmax_normal  = MinMaxScaler().fit_transform(data_normal)
minmax_outlier = MinMaxScaler().fit_transform(data_outlier)
robust_normal  = RobustScaler().fit_transform(data_normal)
robust_outlier = RobustScaler().fit_transform(data_outlier)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# ── Row 0: No outlier ──────────────────────────────────────────────────────────
axes[0, 0].hist(std_normal, bins=20, color='steelblue', edgecolor='white')
axes[0, 0].set_title('StandardScaler\n(no outlier)', fontweight='bold', color='green')
axes[0, 0].set_xlabel('Scaled value')
axes[0, 0].axvline(0, color='red', linestyle='--', linewidth=2)

axes[0, 1].hist(minmax_normal, bins=20, color='steelblue', edgecolor='white')
axes[0, 1].set_title('MinMaxScaler\n(no outlier)', fontweight='bold', color='green')
axes[0, 1].set_xlabel('Scaled value')

axes[0, 2].hist(robust_normal, bins=20, color='steelblue', edgecolor='white')
axes[0, 2].set_title('RobustScaler\n(no outlier)', fontweight='bold', color='green')
axes[0, 2].set_xlabel('Scaled value')
axes[0, 2].axvline(0, color='red', linestyle='--', linewidth=2)

# ── Row 1: With outlier ────────────────────────────────────────────────────────
def plot_zoomed(ax, normal_vals, outlier_val, bar_color, title_color, title,
                annot_color='red', annot_face='#ffe0e0', annot_edge='red', fmt='.1f'):
    ax.hist(normal_vals, bins=20, color=bar_color, edgecolor='white')
    ax.set_xlim(normal_vals.min() - abs(normal_vals.min()) * 0.05,
                normal_vals.max() + abs(normal_vals.max()) * 0.05)
    ax.set_title(title, fontweight='bold', color=title_color)
    label = f'{outlier_val:{fmt}}'
    ax.set_xlabel(f'Scaled value  [outlier at {label}, off-chart →]')
    ax.annotate(f'Outlier at {label} →',
                xy=(1.0, 0.82), xycoords='axes fraction', fontsize=10,
                ha='right', color=annot_color,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=annot_face,
                          edgecolor=annot_edge, alpha=0.9))

plot_zoomed(axes[1, 0], std_outlier[:-1].flatten(), std_outlier[-1][0],
            'salmon', 'red', 'StandardScaler\n(with outlier – crushed near 0!)')
axes[1, 0].axvline(std_outlier[:-1].mean(), color='darkred', linestyle='--',
                   linewidth=2, label='Mean ≈ 0')
axes[1, 0].legend(fontsize=9)

plot_zoomed(axes[1, 1], minmax_outlier[:-1].flatten(), minmax_outlier[-1][0],
            'salmon', 'red', 'MinMaxScaler\n(with outlier – crushed near 0!)')

plot_zoomed(axes[1, 2], robust_outlier[:-1].flatten(), robust_outlier[-1][0],
            'mediumseagreen', 'green', 'RobustScaler\n(with outlier – shape preserved!)',
            annot_color='darkorange', annot_face='#fff3cd', annot_edge='orange', fmt='.0f')
axes[1, 2].axvline(0, color='red', linestyle='--', linewidth=2)

plt.suptitle('Outliers destroy StandardScaler & MinMaxScaler — but not RobustScaler!',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### 🛡️ RobustScaler – Why is it resilient?

$$x_{scaled} = \frac{x - \text{median}(X)}{Q_{75} - Q_{25}}$$

- Uses the **median** instead of the mean → robust to outliers
- Uses the **IQR** instead of standard deviation → unaffected by extreme values

---
# ✂️ Part 5: Train/Test Split

### ⚠️ The Golden Rule: fit only on Train!

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Full Pipeline
np.random.seed(42)
X = pd.DataFrame({
    'age':     np.random.randint(20, 60, 100),
    'salary':  np.random.randint(5000, 30000, 100),
    'country': np.random.choice(['FR', 'DE', 'US'], 100)
})
y = (X['salary'] > 15000).astype(int)  # target: high salary?

# 1. Split FIRST – before any fitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'✂️  Split: {len(X_train)} train, {len(X_test)} test')

# 2. Define preprocessing – fit on TRAIN only!
#    pd.get_dummies on train & test separately is WRONG:
#    → if a category appears only in test, those columns are missing.
#    → reindex is a band-aid; the correct fix is OneHotEncoder fit on train.
preprocessor = ColumnTransformer([
    ('num', StandardScaler(),                       ['age', 'salary']),
    ('cat', OneHotEncoder(drop='first', sparse_output=False,
                          handle_unknown='ignore'), ['country']),
])

X_train_enc = preprocessor.fit_transform(X_train)   # fit + transform on train
X_test_enc  = preprocessor.transform(X_test)         # transform only on test!

# Readable column names
cat_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(['country']).tolist()
col_names = ['age', 'salary'] + cat_cols
X_train_enc = pd.DataFrame(X_train_enc, columns=col_names)
X_test_enc  = pd.DataFrame(X_test_enc,  columns=col_names)

print('\n✅ Train set (first 3 rows):')
display(X_train_enc.head(3).round(3))

print()
print('🔑 Why OneHotEncoder instead of pd.get_dummies?')
print('   get_dummies encodes each set independently → different columns in train vs test!')
print('   OneHotEncoder is fit on train → same columns guaranteed on test.')
print()
print('🔑 Why not fit_transform on the Test set?')
print('   The test set simulates the real world – we cannot peek at its distribution.')
print('   Computing mean/std from test is like reading the exam answers in advance.')


### Lecture Summary:

| Topic | What we learned |
|---|---|
| Missing Data | MCAR / MAR / MNAR – not all missingness is equal |
| Imputation | 6 methods – from Deletion to KNN |
| Encoding | Label vs. One-Hot, avoiding the Dummy Trap |
| Scaling | Standard / MinMax / Robust – choose based on your data |
| Train/Test | fit only on Train – a rule never to break! |

---
*Chen Hajaj | Machine Learning Made Simple*